> ⚠️ **DO NOT RERUN THE CELLS — read this**
>
> This notebook is pre-run with all outputs saved. You can read along without running any cells and see what each step produces.
>
> **Free-tier quota is limited**, and this notebook is designed as a read-only tutorial to get exposed to a few tips and tricks when generating synthetic data.

# Synthetic data for Darija: a case study

You're an ML engineer at a Moroccan startup. Your product needs to understand customer sentiment in **Darija** — Jumia reviews, Avito comments, WhatsApp feedback. But there's no labeled Darija sentiment dataset of any real size. Public scrapes are noisy, unlabeled, and skewed toward social-media venting.

**Your mission in this notebook:** generate ~200 labeled Darija product reviews from scratch, filter them with a stronger model acting as judge, and publish a clean dataset to Hugging Face.

By the end you'll have:

1. A labeled, deduplicated Darija sentiment dataset on the Hub
2. A repeatable recipe to scale it to thousands of samples
3. An eval showing a downstream model can actually recover the labels — i.e. the data *works*

**Tools you'll add to your toolkit along the way:**

- Prompting for low-resource dialects
- Structured output with `response_schema`
- Why naive scale-ups produce homogeneous data
- The hidden dimensions in your data, and how to seed them
- JSON vs. compact output formats and the token bill
- Scaling on free tier (and the batch API for when you have a Paid Tier API key)
- LLM-as-judge for quality filtering
- Publishing with a proper dataset card

Let's go.

## §1 — Setup

We have a few options when it comes to models we can use with AI Studio:

- **`gemma-4-31b-it`**, has the most generous free tier, which is important for the hackathon. Worse than the Gemini 3.x line.
- **`gemini-3-flash-preview`** very good at Darija, minimal difference compared to Gemini 3.1 Pro. Hits the right sweet spot between intelligence and cost effectiveness. Still limited in free tier.
- **`gemini-3.1-pro-preview`** the "smartest" on paper, but not available in free tier.

**On a Paid Tier API key**, you'd typically pick: `gemini-3-flash-preview` for generation (faster, better instruction-following) and `gemini-3.1-pro-preview` for judging (a stronger-than-generator judge gives a more meaningful filter signal). You can swap them in via the form fields below.

You'll need a `GEMINI_API_KEY` saved in Colab Secrets (the 🔑 icon in the left sidebar). Get one free at [aistudio.google.com](https://aistudio.google.com).

In [ ]:
!pip install -q google-genai pydantic datasets huggingface_hub ratelimit tenacity

  Preparing metadata (setup.py) ... done


In [ ]:
import textwrap


def wprint(s: str, width: int = 80) -> None:
    """Wrap text for readable display of multi-line Darija outputs."""
    print(textwrap.fill(s, width=width))

In [ ]:
from google.colab import userdata
from google import genai
from google.genai import types
from google.genai.types import HttpOptions
from pydantic import BaseModel
from typing import Literal
from ratelimit import limits, sleep_and_retry
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception

GENERATOR = "gemini-3-flash-preview"  # @param ["gemma-4-31b-it", "gemini-3-flash-preview"] {allow-input: true}
JUDGE     = "gemini-3-flash-preview"  # @param ["gemini-3-flash-preview", "gemini-3.1-pro-preview"] {allow-input: true}
GEMINI_TIMEOUT_MS = 320000  # @param {type:"integer"}
RATE_LIMIT_RPM = 60        # @param {type:"integer"}

client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY"),
    http_options=HttpOptions(timeout=GEMINI_TIMEOUT_MS),
)

# Retry generate calls on deadline-exceeded / transient errors (up to 3 attempts,
# exponential backoff). Applied as the outermost decorator on every generate function.
_RETRY_KEYWORDS = ("timeout", "deadline", "unavailable")
retry_on_deadline = retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(min=1, max=4),
    retry=retry_if_exception(lambda e: any(k in str(e).lower() for k in _RETRY_KEYWORDS)),
    reraise=True,
)

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
# Sanity check: does the generator work and produce Darija?
resp = client.models.generate_content(
    model=GENERATOR,
    contents="Greet the reader in Moroccan Darija (Arabic script), one short sentence.",
)
print(resp.text)

السلام عليكم، كيدايرين؟


## §2 — The simplest thing that could work

Before scaling anything, get *one* output from the smallest possible prompt. No persona, no scenario, no length budget — just ask the model for a Darija product review and see what comes back.

Why start this minimal? Because every constraint we add later should *earn its place* by fixing a problem we've actually observed. Starting too elaborate hides the failure modes we want students to feel.

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
SIMPLE_PROMPT = "Write a short product review in Moroccan Darija (Arabic script)."

resp = client.models.generate_content(model=GENERATOR, contents=SIMPLE_PROMPT)
print(resp.text)

هذه مراجعة قصيرة بالدارجة المغربية لـ "سماعات بلوتوث" (Wireless Earbuds):

**العنوان: سماعات واعرة بزاف مقابل الثمن!**

"صراحة، هاد السماعات صدموني بالجودة ديالهم. شريتهم هادي سيمانة واستعملتهم بزاف، الصوت نقي والبيس (bass) مجهد، وكتحس براسك معزول على الصداع اللي برا.

أكثر حاجة عجباتني هي الباتري، كيدوز وقت طويل، كنخدم بيهم نهار كامل بلا ما نحتاج نشارجيهم. وحتى الميكرو فالمكالمات خدام مزيان، الناس كيسمعوني بوضوح.

الحاجة الوحيدة اللي شوية ناقصة هي لومبلاج (التعليب) جاني بسيط بزاف، ولكن المهم هو السلعة لداخل. كنصح بيهم أي واحد كيقلب على شي حاجة كاليتي وبثمن مناسب."

---

**Translation (For your reference):**
*   **Title:** Awesome earbuds for the price!
*   **Body:** Honestly, these earbuds shocked me with their quality. I bought them a week ago and used them a lot; the sound is clear, the bass is strong, and you feel isolated from the noise outside. What I liked most is the battery—it lasts a long time; I use them all day without needing to charge. Even the mic works well for calls; p

We get a blob of text without a clear structure — hard to parse. We can't easily use it to build our dataset.

## §3 — Structured output

We could ask the model to print `SENTIMENT: positive` at the end and parse it with a regex. However, it's more robust to take advantage of the SDK's `response_schema` fields which forces the model (either through constrained decoding or just post-generation checks, details unclear) to abide by your schema.

It's as simple as passing a Pydantic schema as `response_schema`. The API validates the output and we get a typed object back — no manual parsing, no broken-format edge cases.

A nice side effect: **the schema's field names are themselves a form of prompting.** We never told the prompt below to "include a sentiment" — the field named `sentiment` does that work.

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
class Review(BaseModel):
    review_darija: str
    sentiment: Literal["positive", "negative"]


# Rate-limited + retried on transient/timeout errors. Both decorators come from §1.
@retry_on_deadline
@sleep_and_retry
@limits(calls=RATE_LIMIT_RPM, period=60)
def generate_review(prompt: str = SIMPLE_PROMPT, model: str = GENERATOR) -> Review:
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Review,
        temperature=0.9,
    )
    resp = client.models.generate_content(model=model, contents=prompt, config=cfg)
    return Review.model_validate_json(resp.text)


review = generate_review()
print(review)

review_darija='هاد الماكينة د القهوة غزالة بزاف، ساهلة في الاستعمال والقهوة ديالها كتجي بنينة.' sentiment='positive'


Notice what just happened: we didn't tell the prompt to produce a sentiment label — the schema field named `sentiment` did. **Field names are prompts.** Name them carefully.

## §4 — Naive scale-up: where does diversity come from?

We have a single generator that produces a labeled review. The obvious next move is to loop it. Let's do that and see what happens.

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
from tqdm import tqdm

reviews = []
for _ in tqdm(range(10)):
    reviews.append(generate_review())

100%|██████████| 10/10 [00:37<00:00,  3.75s/it]


In [ ]:
from collections import Counter

print("Sentiment distribution:")
print(Counter(r.sentiment for r in reviews))

print("\nReviews:")
for r in reviews:
    print(f"  [{r.sentiment[:3]}] {r.review_darija}")

Sentiment distribution:
Counter({'positive': 10})

Reviews:
  [pos] هاد التلفون واعر بزاف، الكاميرا ديالو مجهدة والتصاور كايخرجوا نقيين، غير هو التمن جاني طالع شوية.
  [pos] هاد السبرديلة جاتني زوينة بزاف، الجودة ديالها واعرة و مريحة فالمشي. التوصيل كان فالموعد.
  [pos] هاد التلفون زوين بزاف، الكاميرا ديالو واعرة والباتري كيشد مزيان، غير هو التمن جاني طالع شوية ولكن الجودة كاينة.
  [pos] هاد التلفون زوين بزاف، الكاميرا ديالو خطيرة والشحن كيدوز وقت طويل. التوصيل كان سريع.
  [pos] هاد التليفون واعر بزاف، الكاميرا ديالو طيارة والشارج كيدوز النهار كامل. ولكن الثمن شوية طالع.
  [pos] هاد التيليفون واعر بزاف، خفيف والباتري كادوز نهار كامل. التمن طالع شوية ولكن كيستاهل. التوصيل كان زربان.
  [pos] هاد التيليفون واعر بزاف، الكاميرا ديالو مجهدة والصورة صافية، ولكن الثمن جاني شوية غالي على الجيب.
  [pos] هد الكاسك زوين بزاف، الصوت نقي والثمن مناسب، ولكن التوصيل تعطل عليا شوية.
  [pos] هاد التلفون زوين بزاف، الكاميرا ديالو خطيرة و الباتري كيبقى لنهار كامل. التوصيل كان سريع و الثمن مناسب.
  [pos] ه

A few things we can notice right off the bat:

- **Sentiment collapses to one side.** All 3 positive — the model has a positivity prior for product reviews.
- **The same opening phrases repeat.** Different calls, but the model walks the most-likely path each time. "هاد التليفون واعر بزاف"
- **It's almost always the same product.** A phone, in this instance.

This is the real problem of synthetic data: independent calls from the same prompt have no reason to diverge. The LLM will often follow very similar paths during decoding (even with non-zero temp as we had above).

Two things are missing, and §5 adds both:

1. The model needs to see **different inputs** for each call, so it can't take the same path.
2. We need to decompose the problem into clear constraints and enforce diversity at the prompt-level: what makes a review different from the other? what are the hidden dimensions we could tweak here?

## §5 — The hidden dimensions in a review

Pause and ask: what is a "product review" actually a function of?

- **Product category** — electronics, fashion, beauty, home, baby, food, books
- **Sentiment** — positive, negative, neutral (we want forced balance, not the model's positivity prior)
- **Aspect of focus** — price, quality, shipping, customer service, packaging, sizing, authenticity
- **Reviewer persona** — young urban, middle-aged parent, student, elderly first-time online shopper
- **Region / register** — Casablanca French-mixed, Fes formal, Marrakech informal, Tangier northern
- **Length** — short rant, medium, long story

These were hiding inside §3's tiny `Review` schema all along — the model just picked its defaults for each. Two moves now:

1. **Grow the schema** to record those dimensions in the output, so we can audit them.
2. **Seed the inputs** so each call gets a *different* combination, breaking the most-likely-path collapse.

### Grow the schema

Note we're redefining `Review` in-place — the schema *grew* because the failure in §4 demanded it. We also add a small `Seed` Pydantic and a `GeneratedReview` wrapper so every record carries both *what we asked for* and *what the model produced*.

In [ ]:
class AspectSentiment(BaseModel):
    aspect: Literal["price", "quality", "shipping",
                    "customer_service", "packaging", "sizing", "authenticity"]
    polarity: Literal["positive", "negative", "neutral"]


class Review(BaseModel):
    product_category: Literal["electronics", "fashion", "beauty",
                              "home", "baby", "food", "books"]
    sentiment: Literal["positive", "negative", "neutral"]
    star_rating: int  # 1..5
    review_darija: str
    aspects: list[AspectSentiment]


class Seed(BaseModel):
    category: str
    sentiment: str
    aspect: str
    persona: str
    region: str
    length_range: tuple[int, int]


class GeneratedReview(BaseModel):
    seed: Seed
    review: Review

### Seed the inputs

The schema tells the model **what to output**. The seed tells it **what to be talking about**. We build a grid of dimension values and sample one combination per call.

In [ ]:
import random

CATEGORIES = ["electronics", "fashion", "beauty", "home", "baby", "food", "books"]
SENTIMENTS = ["positive", "negative", "neutral"]
ASPECTS    = ["price", "quality", "shipping", "customer_service",
              "packaging", "sizing", "authenticity"]
PERSONAS   = ["young urban professional in their late 20s",
              "middle-aged parent shopping for the household",
              "university student on a tight budget",
              "first-time online shopper, older than 50"]
REGIONS    = ["Casablanca, with occasional French words",
              "Fes, slightly more formal Darija",
              "Marrakech, very informal",
              "Tangier, with northern Darija flavor"]
LENGTHS    = [(5, 10), (10, 15), (20, 30)]


def random_seed() -> dict:
    return dict(
        category=random.choice(CATEGORIES),
        sentiment=random.choice(SENTIMENTS),
        aspect=random.choice(ASPECTS),
        persona=random.choice(PERSONAS),
        region=random.choice(REGIONS),
        length_range=random.choice(LENGTHS),
    )


def build_prompt(category, sentiment, aspect, persona, region, length_range) -> str:
    lo, hi = length_range
    return f"""You are a {persona} from {region}.
You're writing a product review on Jumia for an item in the **{category}** category.
Your overall feeling is **{sentiment}**, driven mainly by the **{aspect}** of the product.

Write a {lo}-{hi} word review in informal Moroccan Darija (Arabic script).
- First person, mimicking the tone of a review posted on an online platform.
- You can mix in French/Spanish words based on the region.
- Avoid Modern Standard Arabic.
- No title, no star rating in the text, no emoji"""


# Peek at one seeded prompt to see what we're sending
example = random_seed()
print(build_prompt(**example))

You are a first-time online shopper, older than 50 from Marrakech, very informal.
You're writing a product review on Jumia for an item in the **beauty** category.
Your overall feeling is **positive**, driven mainly by the **customer_service** of the product.

Write a 10-15 word review in informal Moroccan Darija (Arabic script).
- First person, mimicking the tone of a review posted on an online platform.
- You can mix in French/Spanish words based on the region.
- Avoid Modern Standard Arabic.
- No title, no star rating in the text, no emoji


### Get K reviews per request

We could now loop 10 standalone calls with random seeds. But each request costs us quota, and we're about to do this 200× in §7 — let's introduce the **batching pattern** here. The trick: ask the model for K reviews under one seeded context in a single request. The schema becomes `response_schema=list[Review]` and we get K typed `Review` objects out.

Why "K reviews per seed" rather than "K different seeds per request"? Because the model can see its own prior generations within the same response and deliberately vary surface details — different products, phrasings, aspect mentions — while staying faithful to the seed. **Seed-level diversity comes from doing many calls** (and the seed grid we just built); **within-seed diversity comes from this batching**. Both matter, and they compose at scale.

In [ ]:
@retry_on_deadline
@sleep_and_retry
@limits(calls=RATE_LIMIT_RPM, period=60)
def generate_review_batch(seed_dict: dict, k: int, model: str = GENERATOR) -> tuple[list[Review], int]:
    """Generate k Reviews from a single seed in one request.

    Returns (reviews, output_tokens). Uses response_schema=list[Review] so the
    model emits a JSON array we parse with Pydantic.
    """
    prompt = build_prompt(**seed_dict) + f"\n\nReturn a JSON array of {k} different reviews for this context. Vary the specific product, wording, and which aspects are mentioned — but keep all reviews faithful to the persona, region, target sentiment, and focus aspect above."
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[Review],
        temperature=0.9,
    )
    resp = client.models.generate_content(model=model, contents=prompt, config=cfg)
    tokens = resp.usage_metadata.candidates_token_count
    # Strip markdown code fences — open models sometimes wrap JSON in ```...```
    resp_text = resp.text.replace("```", "").strip()
    reviews = [Review.model_validate(r) for r in json.loads(resp_text)]
    return reviews, tokens

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
import json

NUM_SEEDS = 5     # @param {type:"integer"}
BATCH_SIZE = 5    # @param {type:"integer"}

seeded_pairs: list[tuple[dict, list[Review]]] = []
seeded_tokens = 0

for _ in tqdm(range(NUM_SEEDS)):
    seed_dict = random_seed()
    reviews, tokens = generate_review_batch(seed_dict, BATCH_SIZE)
    seeded_tokens += tokens
    seeded_pairs.append((seed_dict, reviews))

seeded_reviews = [r for _, batch in seeded_pairs for r in batch]

100%|██████████| 5/5 [01:07<00:00, 13.50s/it]


In [ ]:
print("Sentiment distribution:")
print(Counter(r.sentiment for r in seeded_reviews))

print("\nCategory distribution:")
print(Counter(r.product_category for r in seeded_reviews))

print("\nReviews:")
for r in seeded_reviews:
    tag = f"{r.sentiment[:3]}/{r.product_category[:4]}"
    wprint(f"  [{tag:>10}] {r.review_darija}")

Sentiment distribution:
Counter({'negative': 15, 'positive': 10})

Category distribution:
Counter({'books': 10, 'food': 5, 'fashion': 5, 'electronics': 5})

Reviews:
  [  pos/food] سيرفيس زوين و التعامل كان طوب، تبارك الله
  [  pos/food] الناس ضريفين و السيرفيس كليون كان في المستوى صراحة
  [  pos/food] تعاملهم بروفيسيونيل بزاف و الخدمة كانت ممتازة صراحة
  [  pos/food] شرحو ليا كولشي و الخدمة كانت طوب الصراحة
  [  pos/food] الخدمة ممتازة و هاد الناس كيعرفو يتعاملو مع الكليان
  [  pos/book] كتاب أوريجينال، لوراق نقيين بزاف و لكتيبة باينة. سيت فريمون دو لا
كاليتي.
  [  pos/book] لكتاب أوريجينال مية فالمية، وراقو مزيانين و الكوفيرتور صحيحة.
خديتو ف برومو، طوب.
  [  pos/book] هاد لليفر أوريجينال، ماشي ديك لكوبي لخايبة. تري ساتيسفي ب لخدمة د
لفوندور.
  [  pos/book] الصراحة، لكتاب نقي و لوراق ديالو أوريجينال. جاني ف الوقت، سيت
نيكيل.
  [  pos/book] ما توقعتش لكتاب يجيك أوريجينال بهاد لثمن. لوراق نقيين و لخدمة
بارفيت.
  [  neg/fash] الصباط وصلني في ميكة مقطعة و الكرتونة مبهدلة بزاف حشومة هاد
l

Much better. Sentiment is balanced, categories are spread, openings differ. The seed grid forced the model out of its default trajectory by giving each call genuinely different context to anchor on.

We don't always necessarily want balanced. If we know that our product will have to handle more electronics than beauty products, then it might also make sense to skew the distribution towards that. The point is, this framework gives us the ability to decide which distribution we sample on vs. just relying on the model's biases.

A useful mental model going forward:

> **The schema controls output shape. The seed controls input variety. Both are needed.**

Next (§6) we'll look at one more cost lever — the verbosity of structured output itself, and when it's worth trading off.

## §6 — JSON is robust but verbose

`response_schema` gave us validation for free, but the cost shows up on the bill. Every record re-spells `"product_category":`, `"sentiment":`, `"aspect":`, `"polarity":` etc — and the `aspects[]` list repeats `aspect` and `polarity` field names *per aspect*. For 200 records that's negligible; for the 10k or 100k version you may eventually run, it's real money.

**What we're going to do in this section:** generate the same records using a compact, pipe-delimited format instead of JSON, then compare the **output-token cost** head-to-head against the §5 baseline. The compact path gives up server-side schema enforcement; we want to see what we get back in return.

### JSON baseline

§5's seeded loop already generated 10 reviews through the JSON / `response_schema` path. We tracked the output tokens there as `seeded_tokens` — let's reuse that count rather than burning more quota for the same measurement.

In [ ]:
json_tokens = seeded_tokens   # from the §5 batched loop
json_reviews = seeded_reviews

print(f"JSON path: {json_tokens} output tokens across {len(json_reviews)} reviews (reused from §5)")
print("\nOne JSON record:")
print(json_reviews[0].model_dump_json(indent=2))

JSON path: 3062 output tokens across 25 reviews (reused from §5)

One JSON record:
{
  "product_category": "food",
  "sentiment": "positive",
  "star_rating": 5,
  "review_darija": "سيرفيس زوين و التعامل كان طوب، تبارك الله",
  "aspects": [
    {
      "aspect": "customer_service",
      "polarity": "positive"
    }
  ]
}


### A compact alternative

Pipe-delimited single line — one record per line, five fields, no field names repeated:

```
CATEGORY|SENTIMENT|RATING|REVIEW_TEXT|ASPECTS
```

We give up schema validation: the model can drift, and we have to write a parser. In exchange we drop the per-record overhead of field names, brackets, quotes, and indentation.

In [ ]:
COMPACT_FORMAT_INSTRUCTIONS = """

Format your response as a SINGLE line with five pipe-separated fields:

CATEGORY|SENTIMENT|RATING|REVIEW_TEXT|ASPECTS

- CATEGORY: one of [electronics, fashion, beauty, home, baby, food, books]
- SENTIMENT: one of [positive, negative, neutral]
- RATING: integer 1-5
- REVIEW_TEXT: the Darija review (must not contain a | character)
- ASPECTS: comma-separated `aspect=polarity` pairs; polarity is pos/neg/neu
  aspect ∈ {price, quality, shipping, customer_service, packaging, sizing, authenticity}
  Example: shipping=neg, quality=pos, price=neu

Return ONLY the single line — no preamble, no quotes, no markdown fences."""

POLARITY_MAP = {"pos": "positive", "neg": "negative", "neu": "neutral"}


def parse_compact(line: str) -> Review:
    category, sentiment, rating, text, aspects_str = line.strip().split("|", 4)
    aspects = [
        AspectSentiment(aspect=a.strip(), polarity=POLARITY_MAP[p.strip()])
        for a, p in (pair.split("=") for pair in aspects_str.split(","))
    ]
    return Review(
        product_category=category.strip(),
        sentiment=sentiment.strip(),
        star_rating=int(rating),
        review_darija=text.strip(),
        aspects=aspects,
    )

> 💡 **Demo cell — 5 API calls.** 5 records is enough to see the token ratio; the JSON baseline was 10, but the per-record ratio is what matters.

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
compact_tokens = 0
compact_reviews = []
compact_raw = []
parse_failures = 0
for _ in tqdm(range(5)):
    seed_dict = random_seed()
    prompt = build_prompt(**seed_dict) + COMPACT_FORMAT_INSTRUCTIONS
    cfg = types.GenerateContentConfig(temperature=0.9)
    resp = client.models.generate_content(model=GENERATOR, contents=prompt, config=cfg)
    compact_tokens += resp.usage_metadata.candidates_token_count
    compact_raw.append(resp.text.strip())
    try:
        compact_reviews.append(parse_compact(resp.text))
    except Exception:
        parse_failures += 1

print(f"Compact path: {compact_tokens} output tokens across 5 reviews ({parse_failures} parse failures)")
print("\nOne compact record (raw, before parsing):")
print(compact_raw[0])

100%|██████████| 5/5 [00:37<00:00,  7.54s/it]

Compact path: 181 output tokens across 5 reviews (0 parse failures)

One compact record (raw, before parsing):
baby|negative|1|بزاف هاد الثمن غالي، هادشي راه ماشي raisonnable|price=neg


In [ ]:
# The two baselines have different N (JSON reused §5's 10; compact ran 5),
# so normalize per record before comparing.
json_per = json_tokens / len(json_reviews)
compact_per = compact_tokens / len(compact_raw)
saved_pct = (1 - compact_per / json_per) * 100
print(f"JSON:    {json_per:>6.1f} tokens/record  ({json_tokens} total / {len(json_reviews)} records)")
print(f"Compact: {compact_per:>6.1f} tokens/record  ({compact_tokens} total / {len(compact_raw)} records)")
print(f"Savings: {saved_pct:>5.1f}% on output tokens")

JSON:     122.5 tokens/record  (3062 total / 25 records)
Compact:   36.2 tokens/record  (181 total / 5 records)
Savings:  70.4% on output tokens


### When to use which

| Use **JSON** (with `response_schema`) when… | Use **compact** when… |
|---|---|
| Prototyping, debugging, learning | Scaling to thousands of records |
| You want schema validation for free | You control both ends of the pipeline |
| Records are small or one-off | Records repeat structure (lots of similar fields) |
| Small N — judge passes, eval, spot checks | Bulk generation |

One nuance on the "schema validation for free" row above — it holds *in principle*, but in *practice* you close most of the gap by enumerating allowed values explicitly in the compact prompt (which is exactly what the `aspect ∈ {...}` line in `COMPACT_FORMAT_INSTRUCTIONS` does). The remaining real difference is **enforcement vs. adherence**: JSON makes invalid emissions literally impossible at the API level; compact relies on the model following your prompt. JSON wins where drift is costly; well-specified compact gets close enough most of the time.

One caveat on the measurement: the compact prompt is slightly *longer* on input (the format instructions add ~100 tokens). For total cost, factor both directions in. Output tokens are typically the dominant cost at scale, so the comparison still favours compact — but it's worth being honest about.

§7 will use **compact** for the 200-record bulk run, then **JSON** again in §8 for the judge — small N, robustness matters more than tokens.

## §7 — Scaling to 200 records on free tier

Time to do the actual run. The plan:

1. Use the **seed grid** from §5 for diversity.
2. Use the **compact format** from §6 to keep output tokens (and our quota) low.
3. **Rate-limit** politely so we don't hit free-tier 429s.
4. **Checkpoint to disk** so we can inspect partial progress (note: Colab's `/content/` is ephemeral — a full session disconnect loses it too. Mount Google Drive if you need real durability).
5. **Log parse failures** without crashing — accept N < 200 if the model occasionally breaks format.

We're not retrying or falling back to JSON on parse failure. This is honest about the cost of giving up schema validation in §6: sometimes you lose a row, that's the tradeoff.

In [ ]:
COMPACT_BATCH_INSTRUCTIONS_TEMPLATE = """

Generate {k} DIFFERENT reviews for the seeded context above. They should vary in:
specific product within the category, exact wording, length within the requested
range, and which aspects each review chooses to mention — while all remaining
faithful to the persona, region, target sentiment, and focus aspect.

Format your response as exactly {k} lines, one review per line:

CATEGORY|SENTIMENT|RATING|REVIEW_TEXT|ASPECTS

Same field rules as before:
- CATEGORY: one of [electronics, fashion, beauty, home, baby, food, books]
- SENTIMENT: one of [positive, negative, neutral]
- RATING: integer 1-5
- REVIEW_TEXT: the Darija review (must not contain a | character)
- ASPECTS: comma-separated `aspect=polarity`; polarity in pos/neg/neu
  aspect ∈ {{price, quality, shipping, customer_service, packaging, sizing, authenticity}}

Return ONLY the {k} lines — no headers, no numbering, no preamble, no markdown."""


@retry_on_deadline
@sleep_and_retry
@limits(calls=RATE_LIMIT_RPM, period=60)
def generate_compact_batch(seed_dict: dict, k: int):
    """Generate k compact reviews from a single seed in one call, rate-limited.

    Returns (reviews: list[Review], failures: list[str], tokens: int).
    """
    prompt = build_prompt(**seed_dict) + COMPACT_BATCH_INSTRUCTIONS_TEMPLATE.format(k=k)
    cfg = types.GenerateContentConfig(temperature=0.9)
    resp = client.models.generate_content(model=GENERATOR, contents=prompt, config=cfg)
    tokens = resp.usage_metadata.candidates_token_count

    reviews, failures = [], []
    for line in resp.text.strip().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            reviews.append(parse_compact(line))
        except Exception:
            failures.append(line)
    return reviews, failures, tokens

### Bulk run — `BATCH_SIZE` reviews per request

Calling the API once per review would mean 200 requests, ~20 minutes at our 10-RPM cap. We can amortize: ask the model for `BATCH_SIZE` reviews per request from one seed. With `BATCH_SIZE=5` and `NUM_SEEDS=40`, that's 40 requests producing 200 reviews — about 4 minutes.

There are two ways to batch:

- **`BATCH_SIZE` different seeds per request.** Faster, but each review is still effectively independent — the model doesn't see what it just wrote.
- **One seed, `BATCH_SIZE` reviews per request.** The model sees its prior outputs within the same response and can deliberately vary specifics. Trades a little seed-level coverage per call for stronger within-batch diversity.

We use the second. Seed-level diversity already comes from the §5 grid (we draw a fresh seed every call); what we gain here is variation in product specifics, phrasing, and which aspects each review chooses to focus on, within one seeded context.

As before, each successful record is appended to a JSONL file with `flush()` per row — a kernel restart still leaves you with a partial dataset on disk. Bump `NUM_SEEDS` down if you're just demoing.

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
from pathlib import Path

NUM_SEEDS = 40    # @param {type:"integer"}
BATCH_SIZE = 5    # @param {type:"integer"}
# total reviews ≈ NUM_SEEDS × BATCH_SIZE = 200

OUT_PATH = Path("darija_reviews_raw.jsonl")
OUT_PATH.unlink(missing_ok=True)  # start a fresh run

bulk_reviews: list[GeneratedReview] = []
failures: list[str] = []
total_output_tokens = 0

with OUT_PATH.open("a") as f:
    for _ in tqdm(range(NUM_SEEDS)):
        seed_dict = random_seed()
        reviews, fails, tokens = generate_compact_batch(seed_dict, BATCH_SIZE)
        total_output_tokens += tokens
        failures.extend(fails)
        seed = Seed(**seed_dict)
        for r in reviews:
            bundled = GeneratedReview(seed=seed, review=r)
            bulk_reviews.append(bundled)
            f.write(bundled.model_dump_json() + "\n")
            f.flush()  # durable per-row in case the kernel dies

100%|██████████| 40/40 [14:19<00:00, 21.50s/it]


In [ ]:
print(f"Generated: {len(bulk_reviews)} records  (expected ~{NUM_SEEDS * BATCH_SIZE})")
print(f"Failed:    {len(failures)} parse failures")
print(f"Output tokens used: {total_output_tokens:,}")
print(f"Checkpoint: {OUT_PATH} ({OUT_PATH.stat().st_size:,} bytes)")

print("\n--- Distribution check ---")
print("Sentiment:", Counter(r.review.sentiment for r in bulk_reviews))
print("Category: ", Counter(r.review.product_category for r in bulk_reviews))

print("\n--- Sample successful record ---")
print(bulk_reviews[0].model_dump_json(indent=2))

if failures:
    print("\n--- Sample parse failure (raw model output) ---")
    print(failures[0][:300])

Generated: 195 records  (expected ~200)
Failed:    5 parse failures
Output tokens used: 9,404
Checkpoint: darija_reviews_raw.jsonl (105,276 bytes)

--- Distribution check ---
Sentiment: Counter({'neutral': 80, 'positive': 70, 'negative': 45})
Category:  Counter({'food': 40, 'baby': 40, 'books': 30, 'home': 30, 'fashion': 20, 'electronics': 20, 'beauty': 15})

--- Sample successful record ---
{
  "seed": {
    "category": "food",
    "sentiment": "negative",
    "aspect": "price",
    "persona": "first-time online shopper, older than 50",
    "region": "Casablanca, with occasional French words",
    "length_range": [
      5,
      10
    ]
  },
  "review": {
    "product_category": "food",
    "sentiment": "negative",
    "star_rating": 1,
    "review_darija": "غالي بزاف هاد أتاي الثمن طالع c'est trop",
    "aspects": [
      {
        "aspect": "price",
        "polarity": "negative"
      }
    ]
  }
}

--- Sample parse failure (raw model output) ---
FOOD|neutral|3|خديت هاد القهوة حي

### Optional: Batch API for paid tier

> ⚠️ **This cell will not run on the free tier — `client.batches` requires a paid project.** Keep it as boilerplate for when you have a Paid Tier API key.

For 200 records, sequential is fine. Once you're scaling into the thousands, the [Batch API](https://ai.google.dev/api/batch) is the right tool:

- **~50% discount** on input and output tokens
- **Asynchronous** — submit and come back (up to 24h turnaround)
- **No per-minute rate limiting** within the batch
- One job submission → one results download, instead of N polls

The shape of the call below — same prompts, same compact format, just packaged as one job — is the canonical pattern. Verify exact field names against the current [`google-genai` docs](https://ai.google.dev) before relying on it.

In [ ]:
# DO NOT RUN ON FREE TIER. Illustrative boilerplate for paid-tier scaling.
#
# inline_requests = []
# seed_log = []
# for _ in range(NUM_SEEDS * BATCH_SIZE):
#     seed_dict = random_seed()
#     seed_log.append(seed_dict)
#     prompt = build_prompt(**seed_dict) + COMPACT_FORMAT_INSTRUCTIONS
#     inline_requests.append({"contents": [{"parts": [{"text": prompt}]}]})
#
# batch_job = client.batches.create(
#     model=GENERATOR,
#     src=inline_requests,
#     config={"display_name": "darija-reviews-bulk"},
# )
#
# import time
# while batch_job.state.name not in ("JOB_STATE_SUCCEEDED", "JOB_STATE_FAILED"):
#     time.sleep(30)
#     batch_job = client.batches.get(name=batch_job.name)
#
# # Each inlined response corresponds to inline_requests[i] / seed_log[i].
# # Parse each with parse_compact() and wrap with the matching seed.
# results = client.batches.get(name=batch_job.name).dest.inlined_responses

We now have a few hundred labeled Darija reviews on disk. But "we got the model to produce them" isn't the same as "they're good." §8 introduces a **judge** — a stronger model that scores each record against a quality rubric and surfaces the bad ones.

## §8 — Judging the dataset

> ⚠️ **Real API calls.** This section judges all 200 generated records. With `BATCH_SIZE_JUDGE=50` batching introduced below, that's ~4 requests.

Quality has two distinct levels and we'll address both:

1. **Per-record** — is *this single review* well-formed and correctly labeled? Run by an LLM judge against an explicit rubric.
2. **Dataset-level** — is the *collection as a whole* balanced, calibrated, diverse? Checked with simple aggregate analyses *after* judging. No LLM needed.

Two upfront caveats:

- **Diversity vs. quality is a real tradeoff.** A stricter judge produces a cleaner-looking dataset that's also less diverse — you systematically reject edge cases (sarcasm, mixed sentiment, dialectal extremes) that a real classifier needs to handle. We tag everything, drop nothing, and let downstream consumers decide where to threshold.
- **Same-model judging has known biases.** Ideally the judge is *stronger* than the generator. We're using `gemini-3-flash-preview` for both because of free-tier constraints; the signal is still useful for catching obvious failures but you'd swap `JUDGE` to `gemini-3.1-pro-preview` on a paid project for a more independent verdict.

### The rubric

Six criteria, each scored 1–5, plus an overall score and a one-sentence reason.

In [ ]:
class Verdict(BaseModel):
    # Scores are 1-5 ints. We use plain `int` (not Literal[1..5]) because
    # Gemini's response_schema doesn't accept integer enums — the rubric
    # prompt anchors the range.
    language: int              # Is review_darija clean Moroccan Darija?
    sentiment_alignment: int   # Text matches sentiment label + star_rating sanity
    aspect_justification: int  # Each aspects[] entry justified by the text
    seed_alignment: int        # Matches seeded category + focus aspect
    authenticity: int          # Reads like a real review vs. AI-generated
    specificity: int           # Concrete product details vs. generic praise/complaint
    overall: int               # Holistic score
    reason: str                # One sentence; "no major issues" if overall >= 4


class JudgedReview(BaseModel):
    seed: Seed
    review: Review
    verdict: Verdict

In [ ]:
JUDGE_RUBRIC = """\
Score each review on six criteria, 1-5 (5 = excellent, 1 = poor):

1. LANGUAGE — Is review_darija in clean Moroccan Darija (Arabic script)?
   5: Clean Darija; natural code-mixing with French/Spanish is fine.
   1: MSA, pure French, or wrong dialect entirely.

2. SENTIMENT_ALIGNMENT — Does the text express the sentiment label, and does star_rating make sense?
   5: Unambiguous; star matches (positive=4-5, neutral=3, negative=1-2).
   1: Text contradicts the label, or star is opposite.

3. ASPECT_JUSTIFICATION — For each entry in aspects[], can you find textual support for the aspect AND its polarity?
   5: Every tag clearly supported by the text.
   1: Tags are unrelated to what the review actually says.

4. SEED_ALIGNMENT — Does the review match the seed's category and address the focus aspect?
   5: Clearly about that category; focus aspect is prominent.
   1: Drifted to a different product, or ignored the focus aspect entirely.

5. AUTHENTICITY — Does it read like a real Jumia/Avito review or AI-generated text?
   Tells: structured "Pros:/Cons:" formatting, generic phrasing, hedging language,
   absence of typos, over-coherent paragraphs.
   5: Sounds like a real person typing casually on their phone.
   1: Obviously AI-generated.

6. SPECIFICITY — Does it reference concrete product details, or could it be about anything?
   5: Specific details (price, features, situations).  1: Generic.

Then provide an OVERALL score (1-5) and a one-sentence REASON (or "no major issues" if overall >= 4)."""


def _format_judge_record(i: int, gr: GeneratedReview) -> str:
    return f"""\
=== Record {i} ===
ORIGINAL REQUEST: category={gr.seed.category}, sentiment={gr.seed.sentiment}, aspect={gr.seed.aspect}, persona={gr.seed.persona}, region={gr.seed.region}
GENERATED: product_category={gr.review.product_category}, sentiment={gr.review.sentiment}, star_rating={gr.review.star_rating}, aspects={[(a.aspect, a.polarity) for a in gr.review.aspects]}
review_darija:
\"\"\"
{gr.review.review_darija}
\"\"\"
"""

### Bulk judging

Same batching trick as §7: judge `BATCH_SIZE_JUDGE` reviews per request, returning a JSON array of that many `Verdict` objects. With `BATCH_SIZE_JUDGE=50` and 200 records, that's just 4 requests instead of 200.

If the model returns fewer than `BATCH_SIZE_JUDGE` verdicts or the JSON fails to parse, we fall back to judging that batch one record at a time — preserves correctness at the cost of a few extra calls.

In [ ]:
@retry_on_deadline
@sleep_and_retry
@limits(calls=RATE_LIMIT_RPM, period=60)
def judge_one(gr: GeneratedReview) -> Verdict:
    """Single-record fallback used when batch judging fails to parse cleanly."""
    prompt = f"""You are evaluating one synthetic Moroccan Darija product review.

{JUDGE_RUBRIC}

{_format_judge_record(1, gr)}

Return one Verdict object."""
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Verdict,
        temperature=0.1,
    )
    resp = client.models.generate_content(model=JUDGE, contents=prompt, config=cfg)
    return Verdict.model_validate_json(resp.text)


@retry_on_deadline
@sleep_and_retry
@limits(calls=RATE_LIMIT_RPM, period=60)
def judge_batch(batch: list[GeneratedReview]) -> list[Verdict]:
    """Judge a batch of reviews in one call. Falls back to individual judging on failure."""
    k = len(batch)
    records = "\n".join(_format_judge_record(i + 1, gr) for i, gr in enumerate(batch))
    prompt = f"""You are evaluating {k} synthetic Moroccan Darija product reviews.

{JUDGE_RUBRIC}

RECORDS:
{records}

Return a JSON array of exactly {k} verdict objects, one per record, in the SAME ORDER as listed above."""
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[Verdict],
        temperature=0.1,
    )
    resp = client.models.generate_content(model=JUDGE, contents=prompt, config=cfg)
    try:
        resp_text = resp.text.replace("```", "").strip()
        verdicts = [Verdict.model_validate(v) for v in json.loads(resp_text)]
        if len(verdicts) != k:
            raise ValueError(f"expected {k} verdicts, got {len(verdicts)}")
        return verdicts
    except Exception:
        return [judge_one(gr) for gr in batch]

In [ ]:
# DO NOT RERUN — uses your API quota. Output is saved below.
BATCH_SIZE_JUDGE = 30    # @param {type:"integer"}

JUDGED_PATH = Path("darija_reviews_judged.jsonl")
JUDGED_PATH.unlink(missing_ok=True)

judged: list[JudgedReview] = []
with JUDGED_PATH.open("a") as f, tqdm(total=len(bulk_reviews)) as pbar:
    for i in range(0, len(bulk_reviews), BATCH_SIZE_JUDGE):
        batch = bulk_reviews[i:i + BATCH_SIZE_JUDGE]
        verdicts = judge_batch(batch)
        for gr, verdict in zip(batch, verdicts):
            jr = JudgedReview(seed=gr.seed, review=gr.review, verdict=verdict)
            judged.append(jr)
            f.write(jr.model_dump_json() + "\n")
            f.flush()
            pbar.update(1)

print(f"Judged {len(judged)} records, written to {JUDGED_PATH}")

100%|██████████| 195/195 [02:40<00:00,  1.21it/s]

Judged 195 records, written to darija_reviews_judged.jsonl


### Score distributions

Per-criterion histograms tell us which failure modes are most common.

In [ ]:
import statistics

criteria = ["language", "sentiment_alignment", "aspect_justification",
            "seed_alignment", "authenticity", "specificity", "overall"]

print(f"{'criterion':<22} {'mean':>5} {'med':>4}  distribution (1..5)")
print("-" * 60)
for c in criteria:
    scores = [getattr(j.verdict, c) for j in judged]
    dist = Counter(scores)
    bar = " ".join(f"{dist.get(i, 0):>3}" for i in range(1, 6))
    print(f"{c:<22} {statistics.mean(scores):>5.2f} {statistics.median(scores):>4.1f}  {bar}")

criterion               mean  med  distribution (1..5)
------------------------------------------------------------
language                4.98  5.0    0   0   0   3 192
sentiment_alignment     5.00  5.0    0   0   0   0 195
aspect_justification    4.98  5.0    0   0   0   3 192
seed_alignment          4.98  5.0    0   0   0   4 191
authenticity            4.84  5.0    0   0   0  31 164
specificity             4.43  5.0    0   4  14  72 105
overall                 4.89  5.0    0   0   0  21 174


### What did the judge dislike?

Sample of the lowest-scoring records, with the judge's reason. This is the most instructive cell — read what the judge flagged.

In [ ]:
lowest = sorted(judged, key=lambda j: j.verdict.overall)[:5]
for jr in lowest:
    v = jr.verdict
    print(f"overall={v.overall}  (lang={v.language} sent={v.sentiment_alignment} "
          f"asp={v.aspect_justification} seed={v.seed_alignment} "
          f"auth={v.authenticity} spec={v.specificity})")
    print(f"reason: {v.reason}")
    wprint(f"text:   {jr.review.review_darija}")
    print(f"seed:   category={jr.seed.category} sentiment={jr.seed.sentiment} aspect={jr.seed.aspect}")
    print()

overall=4  (lang=5 sent=5 asp=5 seed=5 auth=4 spec=3)
reason: no major issues
text:   خدمة الزبناء في المستوى المطلوب تعامل راقي وسرعة فالتواصل كتحبب الواحد
يشري من عندكم
seed:   category=fashion sentiment=positive aspect=customer_service

overall=4  (lang=5 sent=5 asp=5 seed=5 auth=4 spec=3)
reason: no major issues
text:   الله يجازيكم بخير على حسن التعامل والاحترافية فهاد الكوموند
seed:   category=books sentiment=positive aspect=customer_service

overall=4  (lang=4 sent=5 asp=5 seed=5 auth=4 spec=4)
reason: The word 'shoba' is likely a typo for 'shoppa' or 'souba', but the Northern flavor is present.
text:   الشوبا وصلتني دغيا ولكن الثوب طلع عادي ماشي شي حاجة اللي مزيونة بزاف
seed:   category=fashion sentiment=neutral aspect=quality

overall=4  (lang=5 sent=5 asp=5 seed=5 auth=4 spec=3)
reason: A bit brief, but fits all criteria.
text:   ليفريزون تعطلات بزاف والحوايج جاو زوينين
seed:   category=baby sentiment=neutral aspect=shipping

overall=4  (lang=5 sent=5 asp=5 seed=5 auth=4 spec

Gemini 3.0 Flash is pretty good, so not a lot of low-hanging fruits to catch.

This section is even more relevant if you're using a weaker model for higher rate limits (e.g. Gemma).

### Dataset-level checks

Per-record judging can't tell you whether the *dataset as a whole* is balanced, calibrated, or diverse. These run in pure Python on the judged dataset — no more LLM calls needed.

In [ ]:
# Class balance — did seed-forced distribution survive parse failures + judging?
print("Sentiment distribution:")
print(" ", Counter(j.review.sentiment for j in judged))
print("\nCategory distribution:")
print(" ", Counter(j.review.product_category for j in judged))
print("\nStar rating distribution:")
print(" ", Counter(j.review.star_rating for j in judged))

# Aspect coverage — across all aspects[] entries in the dataset
all_aspects = [a.aspect for j in judged for a in j.review.aspects]
print(f"\nAspect mentions ({len(all_aspects)} total):")
print(" ", Counter(all_aspects))

Sentiment distribution:
  Counter({'neutral': 80, 'positive': 70, 'negative': 45})

Category distribution:
  Counter({'food': 40, 'baby': 40, 'books': 30, 'home': 30, 'fashion': 20, 'electronics': 20, 'beauty': 15})

Star rating distribution:
  Counter({3: 80, 5: 68, 1: 25, 2: 20, 4: 2})

Aspect mentions (349 total):
  Counter({'quality': 128, 'shipping': 51, 'customer_service': 48, 'price': 40, 'sizing': 33, 'packaging': 25, 'authenticity': 24})


In [ ]:
# Length by sentiment — LLMs frequently write longer positives than negatives.
# Real Jumia reviews are often the opposite (people rant more than they praise).
print("Review length (chars) by sentiment:")
by_sent: dict[str, list[int]] = {}
for j in judged:
    by_sent.setdefault(j.review.sentiment, []).append(len(j.review.review_darija))
for sent, lens in sorted(by_sent.items()):
    print(f"  {sent:<9} n={len(lens):>3}  mean={statistics.mean(lens):>5.0f}  median={statistics.median(lens):>5.0f}")

# Opening n-grams — if classifiers can predict the label from the first few chars,
# they'll overfit to the opener instead of learning sentiment.
print("\nMost repeated 20-char openings (top 5):")
openings = Counter(j.review.review_darija[:20] for j in judged).most_common(5)
for op, n in openings:
    print(f"  ({n}x) {op}")

Review length (chars) by sentiment:
  negative  n= 45  mean=   79  median=   71
  neutral   n= 80  mean=   77  median=   57
  positive  n= 70  mean=  100  median=   84

Most repeated 20-char openings (top 5):
  (2x) الله يجازيكم بخير عل
  (1x) غالي بزاف هاد أتاي ا
  (1x) ثمن الزيت غالي ولكن 
  (1x) هاد الشكلاط غالي بزا
  (1x) العسل مزيان ولكن الث


We now have a tagged, audited dataset. The verdict lives on every record — downstream consumers can filter by `verdict.overall >= 3` (or any other criterion combination they care about) without re-running the judge.

One more move before shipping: in §9 you'll look at what you've produced, tweak the seed dimensions to better fit *your* use case, generate a small custom dataset, and push it to Hugging Face.

## [Optional] §9 — Make it yours

> ⚠️ This is totally optional at this stage if you want to tinker around with the whole e2e process. Your free-tier requests/tokens are precious and might be better spent on the actual project :-)



Look back at what came out — the seeded loop in §5, the §7 bulk run, the lowest-scoring records the judge surfaced in §8. A few things worth asking:

- The judge flagged records on *seed_alignment*. Did the model wander off-category, or weight the focus aspect too lightly? Maybe `build_prompt` needs different phrasing.
- The model kept inventing aspects like `accuracy` (description vs. reality). Worth adding to `ASPECTS`?
- `LENGTHS` tops out at 30 words; real Jumia reviews can run far longer. Add a `(80, 150)` range?
- What's missing from `PERSONAS`? Berber speakers, diaspora, teenagers, shopkeepers?
- Is the model leaning too hard on phones / shoes? Drop `electronics` from `CATEGORIES` for one run and see what fills the gap.

The cells below put every knob in one place — no scrolling back to §5. Tweak something, run the **preview** cell with `K=2`, look at one or two outputs, adjust, repeat. When the previews look right, run the production cell.

In [ ]:
# ── Edit anything below to change what gets generated ────────────────
# These re-bind the names from §5, so generate_review_batch (defined in §5)
# will automatically pick up your new dimensions.

CATEGORIES = ["electronics", "fashion", "beauty", "home", "baby", "food", "books"]
SENTIMENTS = ["positive", "negative", "neutral"]
ASPECTS    = ["price", "quality", "shipping", "customer_service",
              "packaging", "sizing", "authenticity"]
PERSONAS   = [
    "young urban professional in their late 20s",
    "middle-aged parent shopping for the household",
    "university student on a tight budget",
    "first-time online shopper, older than 50",
    # add your own personas here:
]
REGIONS    = [
    "Casablanca, with occasional French words",
    "Fes, slightly more formal Darija",
    "Marrakech, very informal",
    "Tangier, with northern Darija flavor",
    # add your own regions / dialects here:
]
LENGTHS    = [(5, 10), (10, 15), (20, 30)]


def build_prompt(category, sentiment, aspect, persona, region, length_range) -> str:
    lo, hi = length_range
    return f"""You are a {persona} from {region}.
You're writing a product review on Jumia for an item in the **{category}** category.
Your overall feeling is **{sentiment}**, driven mainly by the **{aspect}** of the product.

Write a {lo}-{hi} word review in informal Moroccan Darija (Arabic script).
- First person, mimicking the tone of a review posted on an online platform.
- You can mix in French/Spanish words based on the region.
- Avoid Modern Standard Arabic.
- No title, no star rating in the text, no emoji"""


def random_seed() -> dict:
    return dict(
        category=random.choice(CATEGORIES),
        sentiment=random.choice(SENTIMENTS),
        aspect=random.choice(ASPECTS),
        persona=random.choice(PERSONAS),
        region=random.choice(REGIONS),
        length_range=random.choice(LENGTHS),
    )

### Spot-check: 1–2 examples at a time

Run this cell repeatedly after each tweak above. Cheap to iterate, easy to see whether your changes are landing.

In [ ]:
BATCH_SIZE = 2    # @param {type:"integer"}  — keep low while iterating

seed = random_seed()
reviews, _ = generate_review_batch(seed, BATCH_SIZE)
print("Seed:", seed)
print()
for r in reviews:
    wprint(r.review_darija)
    print(f"  [{r.sentiment} / {r.product_category} / {r.star_rating}★]")
    print()

> **Skipped, but worth doing if you fine-tune:** embedding-based deduplication (e.g. `text-embedding-004` + cosine similarity, drop pairs above ~0.92) reduces noise from near-identical records and usually improves downstream training. We leave it out here to keep the notebook focused.

## §10 — Check your understanding

Use these questions as a self-check or as a mentor's validation checklist. A solid answer to each should reference specific concepts from the notebook, not just memorized facts.

### Prompting for low-resource languages

1. **Dialect and script matter.** Why is it important to explicitly name both the dialect (Moroccan Darija) *and* the script (Arabic) in the prompt? What could go wrong if you only said "write in Darija" or only specified "Arabic script"?

2. **Personas and regions as input variety.** Look back at the PERSONAS and REGIONS lists. Why is "young urban professional in their late 20s" a better seed than just "different person"? What does the specificity do to the output?

### Structured output and schema design

3. **Field names as prompts.** In §3, we never told the prompt to include a sentiment label — the schema field named `sentiment` did. Why does this work? What assumption about LLM behavior does it rely on?

4. **Why aspects matter.** The Review schema includes `aspects: list[AspectSentiment]`. Why is it important to capture *which aspects* the review mentions, beyond just the overall sentiment? (Hint: think about what a downstream classifier needs to learn from the data.)

### The diversity problem and seed-driven generation

5. **Why independent calls collapse.** In §4, three independent calls all generated positive reviews about phones. Why does this happen? (Hint: think about what the model optimizes for on each pass.)

6. **Seed-level vs. within-seed diversity.** We generate K reviews *per seed* rather than K different seeds per request. Explain the difference:
   - Where does seed-level diversity come from?
   - Where does within-seed diversity come from?
   - Why do both matter?

### Output formats and cost

7. **JSON vs. compact tradeoff.** In §6, we compared two formats: JSON (with `response_schema`) vs. pipe-delimited compact. Name the key cost difference and the key robustness difference. When would you pick each?

8. **Call reduction.** This notebook reduced API calls from ~284 to ~93. What's the main strategy we used to achieve that?

### Dataset-level thinking

9. **Length by sentiment.** In §8, we looked at review length split by sentiment. Why is this distribution worth checking? What could it tell you about potential classifier bias?

### Your turn

10. **One improvement.** Looking at the §8 judge output — the lowest-scoring records and the score distributions — suggest *one* concrete change to the prompt or seed dimensions that you think would improve the dataset. Be specific: which dimension would you change, how, and why?